# E1: compare clip and speaker weights

Compare the saved 39-feature and 91-feature models on validation and test data.
Use their full feature sets and selected counts.

- **Clip weights:** each clip has equal influence within its class.
- **Speaker weights:** each speaker has equal total influence within its class.

Both scores weight classes equally. Keep models, counts, and thresholds fixed.
Validation models use training data; test models use training and validation data.


## 1. Load saved predictions

Run both feature studies through test evaluation first.
Check saved results against the 30,000 clips in `data/`: 15,000 positives and 15,000 negatives.
Load test predictions when available; otherwise use the saved models and feature caches.


In [ ]:
from pathlib import Path
import json
import hashlib

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import balanced_accuracy_score, f1_score

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / 'data'
STUDY_DIR = PROJECT_ROOT / 'outputs' / 'e1_feature_study_accent_30000'
POOLS = (39, 91)
MODEL_NAMES = ('logistic_regression', 'svm', 'random_forest')
MODEL_LABELS = {'logistic_regression': 'Logistic Regression', 'svm': 'SVM',
                'random_forest': 'Random Forest'}
SPLITS = ('validation', 'test')
columns = ['source', 'clip_id', 'speaker_id', 'split', 'is_singaporean',
           'accent_group', 'sampling_rate']
parquet_files = sorted(DATA_ROOT.glob('*_subset/part-*.parquet'))
if not parquet_files:
    raise FileNotFoundError(f'No dataset Parquet files found in {DATA_ROOT}')
metadata = pd.concat([pd.read_parquet(path, columns=columns) for path in parquet_files],
                     ignore_index=True)
if metadata.isna().any().any() or metadata['speaker_id'].eq('').any():
    raise ValueError('Missing clip or speaker metadata.')
if not metadata['is_singaporean'].equals(metadata['accent_group'].eq('Singaporean')):
    raise ValueError('Accent groups and labels disagree.')
if not metadata['sampling_rate'].eq(16000).all():
    raise ValueError('Expected 16 kHz audio.')
metadata = metadata.rename(columns={'clip_id': 'source_clip_id'})
metadata['clip_id'] = metadata['source'] + '::' + metadata['source_clip_id']
metadata['speaker_group'] = metadata['source'] + '::' + metadata['speaker_id']
metadata['label'] = metadata['is_singaporean'].astype(np.int64)
metadata['sampling_rate'] = metadata['sampling_rate'].astype(np.int64)
metadata['split'] = metadata['split'].replace({'val': 'validation'})
metadata = metadata.set_index('clip_id', verify_integrity=True).sort_index()
metadata = metadata[['label', 'speaker_group', 'split', 'source', 'source_clip_id',
                     'speaker_id', 'accent_group', 'sampling_rate']]
if not set(metadata['split']) == {'train', 'validation', 'test'}:
    raise ValueError('Expected train, validation, and test splits.')
if (metadata.groupby('speaker_group')['split'].nunique() > 1).any():
    raise ValueError('Speaker overlap between splits.')
if (metadata.groupby('speaker_group')['label'].nunique() > 1).any():
    raise ValueError('A speaker has mixed labels.')
dataset_id = hashlib.sha256(json.dumps(metadata.reset_index().to_dict(orient='records'),
                                      sort_keys=True).encode()).hexdigest()
cache_metadata = json.loads((STUDY_DIR / 'current_cache_metadata.json').read_text())
if cache_metadata['dataset_id'] != dataset_id:
    raise ValueError('Feature cache metadata differs from data/. Rerun the feature studies.')
cache_id = hashlib.sha256(json.dumps(cache_metadata, sort_keys=True).encode()).hexdigest()
identities = {'cache_id': cache_id, 'dataset_id': dataset_id}
studies, prediction_sets, run_rows = {}, {}, []


def score_lookup(records):
    result = {}
    for record in records:
        key = (record['model'], int(record['feature_count']))
        values = (float(record['balanced_accuracy']), float(record['macro_f1']))
        if key in result and result[key] != values:
            raise ValueError(f'Conflicting saved scores: {key}')
        result[key] = values
    return result


for pool in POOLS:
    directory = STUDY_DIR / ('results' if pool == 39 else 'results_91')
    study = joblib.load(directory / 'study_results.joblib')
    if study['dataset_id'] != identities['dataset_id']:
        raise ValueError(f'{pool}: dataset identity differs from data/.')
    if pool == 39:
        expected_cache = identities['cache_id']
    else:
        with np.load(STUDY_DIR / 'acoustic_features_91.npz', allow_pickle=False) as cache:
            expected_cache = str(cache['cache_id'].item())
        if study['base_cache_id'] != identities['cache_id']:
            raise ValueError('91: base cache differs from the current 39-feature cache.')
    if study['cache_id'] != expected_cache:
        raise ValueError(f'{pool}: feature cache identity differs.')
    if set(study['selected_counts']) != set(MODEL_NAMES):
        raise ValueError(f'{pool}: expected all three classifiers.')
    if any(type(k) is not int or not 1 <= k <= pool for k in study['selected_counts'].values()):
        raise ValueError(f'{pool}: invalid selected counts.')

    with np.load(STUDY_DIR / f'acoustic_features_{pool}.npz', allow_pickle=False) as cache:
        if str(cache['cache_id'].item()) != expected_cache:
            raise ValueError(f'{pool}: cache identity differs.')
        if not np.array_equal(cache['clip_ids'], metadata.index.to_numpy()):
            raise ValueError(f'{pool}: cached clips differ from data/.')
        if not np.array_equal(cache['labels'], metadata['label'].to_numpy()) or not np.array_equal(
                cache['groups'], metadata['speaker_group'].to_numpy()):
            raise ValueError(f'{pool}: cached labels or speakers differ from data/.')
        features = cache['features'].copy()
    if 'test_scores' not in study:
        raise ValueError(f'{pool}: run the feature study through test evaluation first.')
    if 'test_predictions' in study:
        test_rows = study['test_predictions']
        test_source = 'study_results.joblib'
    else:
        test_rows = []
        test_idx = np.flatnonzero(metadata['split'].eq('test').to_numpy())
        for model in MODEL_NAMES:
            for count in sorted({pool, study['selected_counts'][model]}):
                saved = joblib.load(directory / f'{model}_{count}.joblib')
                if saved['cache_id'] != expected_cache or saved['dataset_id'] != dataset_id:
                    raise ValueError(f'{pool}, {model}, {count}: saved model identity differs.')
                predictions = saved['model'].predict(features[test_idx][:, saved['columns']])
                test_rows.extend({'model': model, 'feature_count': count,
                    'clip_id': metadata.index[i], 'speaker_group': metadata.iloc[i]['speaker_group'],
                    'true_label': int(metadata.iloc[i]['label']), 'prediction': int(prediction)}
                    for i, prediction in zip(test_idx, predictions))
        test_source = 'saved models and feature cache'
    studies[pool] = study
    for split, rows in [('validation', study['validation_predictions']), ('test', test_rows)]:
        frame = pd.DataFrame(rows)
        if 'split' in frame and not frame['split'].eq(split).all():
            raise ValueError(f'{pool}: incorrect split in {split} predictions.')
        frame['split'] = split
        prediction_sets[(pool, split)] = frame
    run_rows.append({'feature_pool': pool, **study['selected_counts'],
                     'test_prediction_source': test_source})

display(pd.DataFrame(run_rows))


## 2. Check clips, labels, speakers, and scores

Require complete splits without duplicate clips or overlapping speakers.
Check saved test scores against predictions before comparing weights.


In [ ]:
for (pool, split), frame in prediction_sets.items():
    expected = metadata.loc[metadata['split'] == split, ['speaker_group', 'label']].rename(
        columns={'label': 'true_label'}).sort_index()
    required = {(model, count) for model in MODEL_NAMES
                for count in {pool, studies[pool]['selected_counts'][model]}}
    actual_keys = set(zip(frame['model'], frame['feature_count']))
    if actual_keys != required:
        raise ValueError(f'{pool}, {split}: missing or unexpected model/count combinations.')
    references = score_lookup(studies[pool]['test_scores']) if split == 'test' else None
    for (model, count), rows in frame.groupby(['model', 'feature_count']):
        if rows['clip_id'].duplicated().any():
            raise ValueError(f'{pool}, {split}, {model}, {count}: duplicate clips.')
        actual = rows.set_index('clip_id')[['speaker_group', 'true_label']].sort_index()
        if not actual.equals(expected):
            raise ValueError(f'{pool}, {split}, {model}, {count}: clip metadata differs.')
        if set(rows['true_label']) != {0, 1} or not set(rows['prediction']) <= {0, 1}:
            raise ValueError('Expected binary labels and predictions.')
        if references is not None:
            observed = (balanced_accuracy_score(rows['true_label'], rows['prediction']),
                        f1_score(rows['true_label'], rows['prediction'], average='macro'))
            if not np.allclose(observed, references[(model, count)], rtol=0, atol=1e-12):
                raise ValueError(f'{pool}, {model}, {count}: test predictions differ from saved scores.')
print('Checks passed for both pools and both splits.')

## 3. Calculate both scores

For speaker weights, give each clip weight `1 / clips from its speaker`.
Both scores weight classes equally. Show one row when selected and full counts match.


In [ ]:
def calculate_scores(rows):
    clips_per_speaker = rows.groupby('speaker_group')['clip_id'].transform('size')
    weighted = balanced_accuracy_score(rows['true_label'], rows['prediction'],
                                       sample_weight=1.0 / clips_per_speaker)
    ordinary = balanced_accuracy_score(rows['true_label'], rows['prediction'])
    return {'speaker_weighted_balanced_accuracy_pct': 100 * weighted,
            'ordinary_balanced_accuracy_pct': 100 * ordinary,
            'clip_minus_speaker_points': 100 * (ordinary - weighted)}


comparison_rows = []
for pool in POOLS:
    for split in SPLITS:
        frame = prediction_sets[(pool, split)]
        for model in MODEL_NAMES:
            selected = studies[pool]['selected_counts'][model]
            baseline_rows = frame.loc[(frame['model'] == model) & (frame['feature_count'] == pool)]
            baseline = calculate_scores(baseline_rows)
            roles = [('Full / selected', pool)] if selected == pool else [('Full', pool), ('Selected', selected)]
            for role, count in roles:
                rows = frame.loc[(frame['model'] == model) & (frame['feature_count'] == count)]
                scores = calculate_scores(rows)
                comparison_rows.append({'feature_pool': pool, 'split': split,
                    'model': model, 'role': role, 'feature_count': count,
                    'clips': len(rows), 'speakers': rows['speaker_group'].nunique(), **scores,
                    'speaker_weighted_gain_from_full_points': scores['speaker_weighted_balanced_accuracy_pct'] - baseline['speaker_weighted_balanced_accuracy_pct'],
                    'ordinary_gain_from_full_points': scores['ordinary_balanced_accuracy_pct'] - baseline['ordinary_balanced_accuracy_pct']})
comparison = pd.DataFrame(comparison_rows)
for split in SPLITS:
    print(split.title())
    display(comparison.loc[comparison['split'] == split].round(3))

## 4. Speaker statistics for each split

Count each clip once. Speakers with more clips have more influence under clip weights.


In [ ]:
summary_rows = []
for split in SPLITS:
    clips = metadata.loc[metadata['split'] == split]
    speaker_counts = clips.groupby(['label', 'speaker_group']).size().rename('clips').reset_index()
    for label, name in [(0, 'Non-Singapore'), (1, 'Singapore'), (None, 'Total')]:
        counts = speaker_counts['clips'] if label is None else speaker_counts.loc[speaker_counts['label'] == label, 'clips']
        summary_rows.append({'split': split, 'class': name, 'clips': int(counts.sum()),
            'speakers': len(counts), 'speakers_with_one_clip': int((counts == 1).sum()),
            'speakers_with_multiple_clips': int((counts > 1).sum()),
            'average_clips_per_speaker': counts.mean(),
            'clips_from_repeated_speakers': int(counts[counts > 1].sum())})
    print(f'{split.title()}: clips per speaker')
    distribution = pd.crosstab(speaker_counts['clips'], speaker_counts['label']).rename(
        columns={0: 'Non-Singapore', 1: 'Singapore'})
    distribution['Total'] = distribution.sum(axis=1)
    display(distribution)
speaker_summary = pd.DataFrame(summary_rows)
display(speaker_summary.round(2))

## 5. Compare the scores

Show validation and test charts for each feature pool. Labels show balanced accuracy.


In [ ]:
for split in SPLITS:
    fig, axes = plt.subplots(len(POOLS), len(MODEL_NAMES), figsize=(15, 8), sharey=True, squeeze=False)
    width = 0.36
    for row_axes, pool in zip(axes, POOLS):
        for axis, model in zip(row_axes, MODEL_NAMES):
            rows = comparison.loc[(comparison['feature_pool'] == pool)
                & (comparison['split'] == split) & (comparison['model'] == model)]
            positions = np.arange(len(rows))
            ordinary = axis.bar(positions - width / 2, rows['ordinary_balanced_accuracy_pct'],
                                width, label='Equal clip weights', color='steelblue')
            weighted = axis.bar(positions + width / 2, rows['speaker_weighted_balanced_accuracy_pct'],
                                width, label='Equal speaker weights', color='darkorange')
            axis.bar_label(ordinary, fmt='%.2f%%', padding=3, fontsize=8)
            axis.bar_label(weighted, fmt='%.2f%%', padding=3, fontsize=8)
            axis.set_xticks(positions, [f'{r.role}\n({r.feature_count} features)' for r in rows.itertuples()])
            axis.set(title=f'{MODEL_LABELS[model]} | pool {pool}', ylim=(0, 100),
                     ylabel='Balanced accuracy (%)')
            axis.grid(axis='y', alpha=0.2)
            axis.set_axisbelow(True)
    axes[0, 0].legend(fontsize=8, loc='lower left')
    fig.suptitle(f'{split.title()}: equal clip weights and equal speaker weights')
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

## 6. Read and save the comparison

Positive `clip_minus_speaker_points` means clip weighting scores higher.
Gain columns compare selected models with their full-feature baselines.
Speaker weights change scoring only; source effects and repeated clips remain.
Keep model and count choices fixed after testing.


In [ ]:
for pool in POOLS:
    for split in SPLITS:
        rows = comparison.loc[(comparison['feature_pool'] == pool) & (comparison['split'] == split)]
        largest = rows.loc[rows['clip_minus_speaker_points'].abs().idxmax()]
        print(f"Pool {pool}, {split}: largest clip-minus-speaker difference is "
              f"{largest['clip_minus_speaker_points']:+.2f} percentage points "
              f"for {MODEL_LABELS[largest['model']]} ({largest['feature_count']} features).")
        for model in MODEL_NAMES:
            selected = rows.loc[(rows['model'] == model) & (rows['role'] == 'Selected')]
            if selected.empty:
                continue
            result = selected.iloc[0]
            print(f"  {MODEL_LABELS[model]}: selected-minus-full change is "
                  f"{result['ordinary_gain_from_full_points']:+.2f} points with clip weights and "
                  f"{result['speaker_weighted_gain_from_full_points']:+.2f} points with speaker weights.")

RESULTS_DIR = STUDY_DIR / 'speaker_weight_comparison'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(RESULTS_DIR / 'score_comparison.csv', index=False)
speaker_summary.to_csv(RESULTS_DIR / 'speaker_summary.csv', index=False)
(RESULTS_DIR / 'comparison_sources.json').write_text(json.dumps({
    'dataset_id': identities['dataset_id'],
    'studies': {str(pool): {'cache_id': studies[pool]['cache_id'],
        'selected_counts': studies[pool]['selected_counts'],
        'test_prediction_source': run_rows[i]['test_prediction_source']}
        for i, pool in enumerate(POOLS)},
}, indent=2) + '\n')
print('Saved comparison:', RESULTS_DIR)